In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
dbutils.widgets.text("container", "raw")
dbutils.widgets.text("catalogo", "catalog_au")
dbutils.widgets.text("esquema", "bronze")
dbutils.widgets.text("storageName", "adlsdiegoadama2204")

In [0]:
container = dbutils.widgets.get("container")
catalogo = dbutils.widgets.get("catalogo")
esquema = dbutils.widgets.get("esquema")
storageName = dbutils.widgets.get("storageName")

ruta = f"abfss://{container}@{storageName}.dfs.core.windows.net/clientes_example.csv"

In [0]:
df_circuits = spark.read.option('header', True)\
                        .option('inferSchema', True)\
                        .csv(ruta)

In [0]:
circuits_schema = StructType(fields=[StructField("id_cliente", StringType(), False),
                                     StructField("nombre_completo", StringType(), True),
                                     StructField("fecha_nacimiento", StringType(), True),
                                     StructField("correo", StringType(), True),
                                     StructField("ciudad", StringType(), True),
                                     StructField("fecha_registro",DateType() , True)
                                     
])

In [0]:
df_circuits_final = spark.read\
.option('header', True)\
.schema(circuits_schema)\
.csv(ruta)

In [0]:
circuits_selected_df = df_circuits_final.select(col("id_cliente"), 
                                                col("nombre_completo"), 
                                                col("fecha_nacimiento"), 
                                                col("correo"), 
                                                col("ciudad"), 
                                                col("fecha_registro"))

In [0]:
circuits_selected_df.write.mode("overwrite").insertInto(f"{catalogo}.{esquema}.clientes")

In [0]:
%sql
select * from catalog_au.bronze.clientes